In [0]:
# Criação do schema da camada Silver

spark.sql("CREATE SCHEMA IF NOT EXISTS mvp_1.silver")

DataFrame[]

In [0]:
from pyspark.sql import functions as F

In [0]:
# Verificação da fonte

df_tvd1_municipio = spark.table("mvp_1.bronze.tvd1_municipio")

display(df_tvd1_municipio.limit(5))

Município,1999,2000,2001,2002,2003,2004,2005,2006,2007,2008,2009,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,Total
110001 ALTA FLORESTA D\'OESTE,0,"10,21","91,44","90,02","111,54","97,46","96,02","111,13","146,45","112,81","106,94","102,06","98,97","90,23","100,00","116,58","103,93","103,52","112,32","103,68","136,44","99,72","99,18","123,12","99,43"
110002 ARIQUEMES,0,"22,46","128,40","128,31","155,59","120,76","127,45","124,32","169,74","89,93","100,53","93,29","96,76","111,00","134,58","155,16","103,39","106,01","96,66","94,71","103,94","93,10","76,11","86,07","107,95"
110003 CABIXI,0,"40,78","100,70","90,78","117,02","273,20","103,33","108,57","98,97","90,32","125,45","91,82","85,45","76,36","74,44","126,56","137,84","121,33","158,67","200,00","180,60","120,90","82,50","85,51","110,78"
110004 CACOAL,0,"19,24","97,79","103,62","115,58","102,59","106,54","106,21","107,00","101,00","100,15","102,58","88,09","98,41","98,24","102,50","108,86","101,30","96,02","106,50","106,40","102,75","129,48","103,92","99,27"
110005 CEREJEIRAS,0,"17,56","117,94","102,95","129,17","161,30","102,31","132,05","97,53","110,69","126,27","121,66","118,89","142,40","105,31","129,80","107,75","110,04","104,42","105,70","124,19","93,86","117,30","99,63","109,16"


Os dados referentes a cobertura vacinal não foram agregados por média aritmética, uma vez que os territórios possuem populações alvos diferentes. Será utilizado os indicadores disponíveis diretamente do DataSUS.

In [0]:
from pyspark.sql import functions as F

def transformar_municipio(nome_tabela):
    
    # Lê a tabela Bronze
    df = spark.table(f"mvp_1.bronze.{nome_tabela}")
    
    # Separa código e nome do município
    df = (
        df
        .withColumn(
            "codigo_municipio",
            F.regexp_extract(F.col("Município"), r"^(\d{6})", 1)
        )
        .withColumn(
            "municipio",
            F.trim(
                F.regexp_replace(F.col("Município"), r"^\d{6}\s*", "")
            )
        )
        .withColumn(
            "codigo_uf",
            F.substring(F.col("codigo_municipio"), 1, 2)
        )
    )
    
    # Identifica as colunas de anos
    colunas_anos = [c for c in df.columns if c.isdigit()]
    
    # Transforma anos de colunas para linhas
    df = (
        df
        .unpivot(
            ids=["codigo_municipio", "municipio", "codigo_uf"],
            values=colunas_anos,
            variableColumnName="ano",
            valueColumnName="cobertura"
        )
        .withColumn("ano", F.col("ano").cast("int"))
        .withColumn(
            "cobertura",
            F.regexp_replace(
                F.col("cobertura"), ",", "."
            ).cast("double")
        )
    )
    
    return df

In [0]:
# Transformação das três vacinas em nível municipal

df_tvd1_municipio = transformar_municipio("tvd1_municipio")
df_tvd2_municipio = transformar_municipio("tvd2_municipio")
df_scrvz_municipio = transformar_municipio("scrvz_municipio")

In [0]:
display(df_tvd1_municipio.limit(10))
display(df_tvd2_municipio.limit(10))
display(df_scrvz_municipio.limit(10))

codigo_municipio,municipio,codigo_uf,ano,cobertura
110001,ALTA FLORESTA D\'OESTE,11,1999,0.0
110002,ARIQUEMES,11,1999,0.0
110003,CABIXI,11,1999,0.0
110004,CACOAL,11,1999,0.0
110005,CEREJEIRAS,11,1999,0.0
110006,COLORADO DO OESTE,11,1999,0.0
110007,CORUMBIARA,11,1999,0.0
110008,COSTA MARQUES,11,1999,0.0
110009,ESPIGAO D\'OESTE,11,1999,0.0
110010,GUAJARA-MIRIM,11,1999,0.0


codigo_municipio,municipio,codigo_uf,ano,cobertura
110001,ALTA FLORESTA D\'OESTE,11,2013,83.17
110002,ARIQUEMES,11,2013,63.08
110003,CABIXI,11,2013,38.89
110004,CACOAL,11,2013,80.61
110005,CEREJEIRAS,11,2013,106.19
110006,COLORADO DO OESTE,11,2013,89.45
110007,CORUMBIARA,11,2013,94.9
110008,COSTA MARQUES,11,2013,24.12
110009,ESPIGAO D\'OESTE,11,2013,71.77
110010,GUAJARA-MIRIM,11,2013,55.65


codigo_municipio,municipio,codigo_uf,ano,cobertura
110001,ALTA FLORESTA D\'OESTE,11,2013,39.45
110002,ARIQUEMES,11,2013,32.05
110003,CABIXI,11,2013,32.22
110004,CACOAL,11,2013,51.8
110005,CEREJEIRAS,11,2013,34.07
110006,COLORADO DO OESTE,11,2013,51.48
110007,CORUMBIARA,11,2013,44.9
110008,COSTA MARQUES,11,2013,11.28
110009,ESPIGAO D\'OESTE,11,2013,26.01
110010,GUAJARA-MIRIM,11,2013,25.31


In [0]:
# Validação das tabelas municipais transformadas

for nome, df in {
    "TVD1": df_tvd1_municipio,
    "TVD2": df_tvd2_municipio,
    "SCRVZ": df_scrvz_municipio
}.items():

    print(f"\n{nome}")
    print(f"Registros: {df.count()}")
    print(f"Municípios distintos: {df.select('codigo_municipio').distinct().count()}")

    df.select(
        F.min("ano").alias("ano_inicial"),
        F.max("ano").alias("ano_final"),
        F.min("cobertura").alias("cobertura_min"),
        F.max("cobertura").alias("cobertura_max")
    ).show()


TVD1
Registros: 133728
Municípios distintos: 5572
+-----------+---------+-------------+-------------+
|ano_inicial|ano_final|cobertura_min|cobertura_max|
+-----------+---------+-------------+-------------+
|       1999|     2022|          0.0|       9550.0|
+-----------+---------+-------------+-------------+


TVD2
Registros: 55710
Municípios distintos: 5571
+-----------+---------+-------------+-------------+
|ano_inicial|ano_final|cobertura_min|cobertura_max|
+-----------+---------+-------------+-------------+
|       2013|     2022|          0.0|       7400.0|
+-----------+---------+-------------+-------------+


SCRVZ
Registros: 55710
Municípios distintos: 5571
+-----------+---------+-------------+-------------+
|ano_inicial|ano_final|cobertura_min|cobertura_max|
+-----------+---------+-------------+-------------+
|       2013|     2022|          0.0|       7250.0|
+-----------+---------+-------------+-------------+



In [0]:
# Verificar códigos vazios e linha Total

for nome, df in {
    "TVD1": df_tvd1_municipio,
    "TVD2": df_tvd2_municipio,
    "SCRVZ": df_scrvz_municipio
}.items():

    problemas = df.filter(
        (F.col("codigo_municipio") == "") |
        F.col("codigo_municipio").isNull() |
        (F.upper(F.col("municipio")) == "TOTAL")
    ).count()

    print(f"{nome}: {problemas} registros problemáticos")

TVD1: 24 registros problemáticos
TVD2: 10 registros problemáticos
SCRVZ: 10 registros problemáticos


In [0]:
# Mostrar os registros problemáticos da TVD1

df_tvd1_municipio.filter(
    (F.col("codigo_municipio") == "") |
    F.col("codigo_municipio").isNull()
).select(
    "codigo_municipio",
    "municipio",
    "codigo_uf",
    "ano",
    "cobertura"
).show(30, truncate=False)

+----------------+---------+---------+----+---------+
|codigo_municipio|municipio|codigo_uf|ano |cobertura|
+----------------+---------+---------+----+---------+
|                |Total    |         |1999|6692.1   |
|                |Total    |         |2000|77.5     |
|                |Total    |         |2001|88.43    |
|                |Total    |         |2002|96.92    |
|                |Total    |         |2003|112.95   |
|                |Total    |         |2004|110.93   |
|                |Total    |         |2005|106.55   |
|                |Total    |         |2006|105.35   |
|                |Total    |         |2007|106.8    |
|                |Total    |         |2008|99.81    |
|                |Total    |         |2009|103.74   |
|                |Total    |         |2010|99.93    |
|                |Total    |         |2011|102.39   |
|                |Total    |         |2012|99.5     |
|                |Total    |         |2013|107.46   |
|                |Total    |

Tratamento de registros agregados: As tabelas disponibilizadas pelo DATASUS possuem uma linha denominada Total, que representa um indicador agregado e não um município. Esses registros serão removidos por entender que não cabe na análise.

In [0]:
# Remove registros agregados das tabelas municipais
def remover_total_municipio(df):
    return df.filter(
        F.col("codigo_municipio").rlike(r"^\d{6}$")
    )

In [0]:
df_tvd1_municipio = remover_total_municipio(df_tvd1_municipio)
df_tvd2_municipio = remover_total_municipio(df_tvd2_municipio)
df_scrvz_municipio = remover_total_municipio(df_scrvz_municipio)

In [0]:
for nome, df in {
    "TVD1": df_tvd1_municipio,
    "TVD2": df_tvd2_municipio,
    "SCRVZ": df_scrvz_municipio
}.items():

    print(f"{nome}: {df.count()} registros")

TVD1: 133704 registros
TVD2: 55700 registros
SCRVZ: 55700 registros


In [0]:
# Inspeção das tabelas Bronze por UF

for tabela in ["tvd1_uf", "tvd2_uf", "scrvz_uf"]:
    print(f"\n{tabela}")
    df = spark.table(f"mvp_1.bronze.{tabela}")
    print(df.columns)
    display(df.limit(5))


tvd1_uf
['Unidade_da_Federação', '1999', '2000', '2001', '2002', '2003', '2004', '2005', '2006', '2007', '2008', '2009', '2010', '2011', '2012', '2013', '2014', '2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022', 'Total']


Unidade_da_Federação,1999,2000,2001,2002,2003,2004,2005,2006,2007,2008,2009,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,Total
11 Rondônia,0,"17,15","93,85","94,94","115,24","130,63","115,99","125,36","116,70","103,49","102,29","100,41","102,72","105,40","106,52","146,88","109,00","109,79","103,01","101,65","106,42","84,21","82,54","89,16","102,16"
12 Acre,0,"27,55","67,82","104,56","99,40","123,22","95,12","90,84","109,45","94,17","103,45","96,87","105,29","90,28","95,00","99,20","84,21","75,71","75,14","83,11","87,39","60,15","60,20","70,47","87,21"
13 Amazonas,0,"14,84","27,69","71,27","91,30","86,06","104,03","103,37","110,07","101,56","103,13","100,11","94,67","103,39","98,77","114,36","95,42","83,56","79,83","89,81","92,12","77,00","73,11","78,94","86,79"
14 Roraima,0,"25,35","61,93","75,26","110,22","110,32","116,01","94,23","97,31","95,51","100,61","94,49","97,98","87,83","89,07","110,16","108,45","90,77","86,53","99,32","81,21","69,50","67,31","66,95","87,68"
15 Pará,0,"10,14","47,94","120,34","124,98","143,10","116,35","113,05","118,49","111,83","117,62","110,95","109,25","102,20","98,49","115,73","71,92","69,61","67,51","77,30","82,81","62,39","62,71","67,46","92,57"



tvd2_uf
['Unidade_da_Federação', '2013', '2014', '2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022', 'Total']


Unidade_da_Federação,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,Total
11 Rondônia,"74,56","112,73","94,61","94,32","81,58","78,87","82,25","64,44","42,14","48,34","77,47"
12 Acre,"27,08","61,61","51,69","64,20","57,00","71,92","78,65","41,58","25,95","37,33","51,62"
13 Amazonas,"56,09","84,51","78,17","75,60","61,32","78,00","82,29","52,17","44,70","48,78","66,24"
14 Roraima,"23,02","89,67","92,42","83,50","86,27","88,27","86,72","65,41","35,51","38,11","67,59"
15 Pará,"34,76","65,35","45,78","62,34","54,16","59,97","71,34","54,79","27,72","30,19","50,72"



scrvz_uf
['Unidade_da_Federação', '2013', '2014', '2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022', 'Total']


Unidade_da_Federação,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,Total
11 Rondônia,"38,30","112,79","94,63","94,97","76,25","53,43","70,74","47,65","1,51","9,81","61,16"
12 Acre,"12,84","59,30","49,30","64,45","52,16","59,06","75,68","34,11","1,63","3,64","41,89"
13 Amazonas,"30,44","84,84","77,43","75,80","59,11","58,52","72,71","34,43","1,31","12,62","51,73"
14 Roraima,"18,29","89,83","92,53","83,64","84,65","71,34","74,53","42,46","1,65","5,48","54,45"
15 Pará,"20,14","57,71","37,78","62,39","51,04","48,29","59,90","46,49","2,55","4,96","39,75"


In [0]:
# Função para transformar tabelas de cobertura por UF

def transformar_uf(nome_tabela):

    # Lê a tabela Bronze
    df = spark.table(f"mvp_1.bronze.{nome_tabela}")

    # Separa código e nome da UF
    df = (
        df
        .withColumn(
            "codigo_uf",
            F.regexp_extract(F.col("Unidade_da_Federação"), r"^(\d{2})", 1)
        )
        .withColumn(
            "uf",
            F.trim(
                F.regexp_replace(
                    F.col("Unidade_da_Federação"),
                    r"^\d{2}\s*",
                    ""
                )
            )
        )
    )

    # Identifica somente as colunas que representam anos
    colunas_anos = [c for c in df.columns if c.isdigit()]

    # Transforma os anos de colunas para linhas
    df = df.unpivot(
        ids=["codigo_uf", "uf"],
        values=colunas_anos,
        variableColumnName="ano",
        valueColumnName="cobertura"
    )

    # Ajusta os tipos
    df = (
        df
        .withColumn("ano", F.col("ano").cast("int"))
        .withColumn(
            "cobertura",
            F.regexp_replace(
                F.col("cobertura"), ",", "."
            ).cast("double")
        )
    )

    return df

In [0]:
# Transformação das tabelas por UF

df_tvd1_uf = transformar_uf("tvd1_uf")
df_tvd2_uf = transformar_uf("tvd2_uf")
df_scrvz_uf = transformar_uf("scrvz_uf")

In [0]:
display(df_tvd1_uf.limit(10))

codigo_uf,uf,ano,cobertura
11,Rondônia,1999,0.0
12,Acre,1999,0.0
13,Amazonas,1999,0.0
14,Roraima,1999,0.0
15,Pará,1999,0.0
16,Amapá,1999,0.0
17,Tocantins,1999,0.0
21,Maranhão,1999,0.0
22,Piauí,1999,0.0
23,Ceará,1999,0.0


In [0]:
Será mantido apenas os registros com código UF válido

In [0]:
# Remove registros que não representam uma UF

def remover_total_uf(df):
    return df.filter(
        F.col("codigo_uf").rlike(r"^\d{2}$")
    )

df_tvd1_uf = remover_total_uf(df_tvd1_uf)
df_tvd2_uf = remover_total_uf(df_tvd2_uf)
df_scrvz_uf = remover_total_uf(df_scrvz_uf)

In [0]:
for nome, df in {
    "TVD1 UF": df_tvd1_uf,
    "TVD2 UF": df_tvd2_uf,
    "SCRVZ UF": df_scrvz_uf
}.items():

    print(
        f"{nome}: "
        f"{df.count()} registros | "
        f"{df.select('codigo_uf').distinct().count()} UFs"
    )

TVD1 UF: 648 registros | 27 UFs
TVD2 UF: 270 registros | 27 UFs
SCRVZ UF: 270 registros | 27 UFs


In [0]:
# Função para transformar tabelas de cobertura por Região

def transformar_regiao(nome_tabela):

    # Lê a tabela Bronze
    df = spark.table(f"mvp_1.bronze.{nome_tabela}")

    # Separa código e nome da região
    df = (
        df
        .withColumn(
            "codigo_regiao",
            F.regexp_extract(F.col("Região"), r"^(\d)", 1)
        )
        .withColumn(
            "regiao",
            F.trim(
                F.regexp_replace(
                    F.col("Região"),
                    r"^\d\s*Região\s*",
                    ""
                )
            )
        )
    )

    # Identifica somente as colunas de anos
    colunas_anos = [c for c in df.columns if c.isdigit()]

    # Transforma anos de colunas para linhas
    df = df.unpivot(
        ids=["codigo_regiao", "regiao"],
        values=colunas_anos,
        variableColumnName="ano",
        valueColumnName="cobertura"
    )

    # Ajusta tipos
    df = (
        df
        .withColumn("ano", F.col("ano").cast("int"))
        .withColumn(
            "cobertura",
            F.regexp_replace(
                F.col("cobertura"), ",", "."
            ).cast("double")
        )
    )

    return df

In [0]:
df_tvd1_regiao = transformar_regiao("tvd1_regiao")
df_tvd2_regiao = transformar_regiao("tvd2_regiao")
df_scrvz_regiao = transformar_regiao("scrvz_regiao")

display(df_tvd1_regiao.limit(10))

codigo_regiao,regiao,ano,cobertura
1,Norte,1999,0.0
2,Nordeste,1999,0.0
3,Sudeste,1999,0.0
4,Sul,1999,0.0
5,Centro-Oeste,1999,525.47
,Total,1999,6692.1
1,Norte,2000,12.07
2,Nordeste,2000,65.03
3,Sudeste,2000,97.54
4,Sul,2000,87.6


Remoção do Total, uma vez que entendido que não fará parte da análise

In [0]:
# Remove o registro agregado "Total" das tabelas regionais

def remover_total_regiao(df):
    return df.filter(
        F.col("codigo_regiao").isin("1", "2", "3", "4", "5")
    )

df_tvd1_regiao = remover_total_regiao(df_tvd1_regiao)
df_tvd2_regiao = remover_total_regiao(df_tvd2_regiao)
df_scrvz_regiao = remover_total_regiao(df_scrvz_regiao)

In [0]:
for nome, df in {
    "TVD1 Região": df_tvd1_regiao,
    "TVD2 Região": df_tvd2_regiao,
    "SCRVZ Região": df_scrvz_regiao
}.items():

    print(
        f"{nome}: "
        f"{df.count()} registros | "
        f"{df.select('codigo_regiao').distinct().count()} regiões"
    )

TVD1 Região: 120 registros | 5 regiões
TVD2 Região: 50 registros | 5 regiões
SCRVZ Região: 50 registros | 5 regiões


Validação final da camada Silver

Verificar a qualidade da camada silver, para garantir bons dados para as análises.

In [0]:
from pyspark.sql import functions as F

tabelas_silver = {
    "tvd1_municipio": (df_tvd1_municipio, "codigo_municipio"),
    "tvd2_municipio": (df_tvd2_municipio, "codigo_municipio"),
    "scrvz_municipio": (df_scrvz_municipio, "codigo_municipio"),

    "tvd1_uf": (df_tvd1_uf, "codigo_uf"),
    "tvd2_uf": (df_tvd2_uf, "codigo_uf"),
    "scrvz_uf": (df_scrvz_uf, "codigo_uf"),

    "tvd1_regiao": (df_tvd1_regiao, "codigo_regiao"),
    "tvd2_regiao": (df_tvd2_regiao, "codigo_regiao"),
    "scrvz_regiao": (df_scrvz_regiao, "codigo_regiao")
}

for nome, (df, codigo) in tabelas_silver.items():

    nulos = df.filter(
        F.col(codigo).isNull() |
        (F.col(codigo) == "") |
        F.col("ano").isNull() |
        F.col("cobertura").isNull()
    ).count()

    duplicados = (
        df.groupBy(codigo, "ano")
          .count()
          .filter(F.col("count") > 1)
          .count()
    )

    periodo = df.agg(
        F.min("ano").alias("inicio"),
        F.max("ano").alias("fim")
    ).first()

    print(
        f"{nome}: "
        f"{df.count()} registros | "
        f"nulos={nulos} | "
        f"duplicados={duplicados} | "
        f"período={periodo['inicio']}-{periodo['fim']}"
    )

tvd1_municipio: 133704 registros | nulos=0 | duplicados=0 | período=1999-2022
tvd2_municipio: 55700 registros | nulos=0 | duplicados=0 | período=2013-2022
scrvz_municipio: 55700 registros | nulos=0 | duplicados=0 | período=2013-2022
tvd1_uf: 648 registros | nulos=0 | duplicados=0 | período=1999-2022
tvd2_uf: 270 registros | nulos=0 | duplicados=0 | período=2013-2022
scrvz_uf: 270 registros | nulos=0 | duplicados=0 | período=2013-2022
tvd1_regiao: 120 registros | nulos=0 | duplicados=0 | período=1999-2022
tvd2_regiao: 50 registros | nulos=0 | duplicados=0 | período=2013-2022
scrvz_regiao: 50 registros | nulos=0 | duplicados=0 | período=2013-2022


In [0]:
#Salvando tabelas na camada silver

tabelas_para_salvar = {
    "tvd1_municipio": df_tvd1_municipio,
    "tvd2_municipio": df_tvd2_municipio,
    "scrvz_municipio": df_scrvz_municipio,

    "tvd1_uf": df_tvd1_uf,
    "tvd2_uf": df_tvd2_uf,
    "scrvz_uf": df_scrvz_uf,

    "tvd1_regiao": df_tvd1_regiao,
    "tvd2_regiao": df_tvd2_regiao,
    "scrvz_regiao": df_scrvz_regiao
}

for nome_tabela, df in tabelas_para_salvar.items():

    (
        df.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(f"mvp_1.silver.{nome_tabela}")
    )

    print(f"{nome_tabela} salva com sucesso.")

tvd1_municipio salva com sucesso.
tvd2_municipio salva com sucesso.
scrvz_municipio salva com sucesso.
tvd1_uf salva com sucesso.
tvd2_uf salva com sucesso.
scrvz_uf salva com sucesso.
tvd1_regiao salva com sucesso.
tvd2_regiao salva com sucesso.
scrvz_regiao salva com sucesso.


In [0]:
display(
    spark.sql("SHOW TABLES IN mvp_1.silver")
)

database,tableName,isTemporary
silver,scrvz_municipio,false
silver,scrvz_regiao,false
silver,scrvz_uf,false
silver,tvd1_municipio,false
silver,tvd1_regiao,false
silver,tvd1_uf,false
silver,tvd2_municipio,false
silver,tvd2_regiao,false
silver,tvd2_uf,false


In [0]:
for tabela in [
    "tvd1_municipio",
    "tvd2_municipio",
    "scrvz_municipio",
    "tvd1_uf",
    "tvd2_uf",
    "scrvz_uf",
    "tvd1_regiao",
    "tvd2_regiao",
    "scrvz_regiao"
]:

    quantidade = spark.table(
        f"mvp_1.silver.{tabela}"
    ).count()

    print(f"{tabela}: {quantidade} registros")

tvd1_municipio: 133704 registros
tvd2_municipio: 55700 registros
scrvz_municipio: 55700 registros
tvd1_uf: 648 registros
tvd2_uf: 270 registros
scrvz_uf: 270 registros
tvd1_regiao: 120 registros
tvd2_regiao: 50 registros
scrvz_regiao: 50 registros
